# Evaluation using denseNet121 model

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications import MobileNetV3Small,DenseNet121,InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, log_loss, jaccard_score
import numpy as np
import os
from PIL import Image
from shutil import copyfile  
import pandas as pd
import seaborn as sns
import cv2
import matplotlib.pyplot as plt

# **Dataset Describtion**
* train - 70%
* test - 20%
* validation - 10%

# Preprocess of the dataset 

# Path of the dataset

In [2]:
# Define the paths to our train, validation, and test datasets
train_data_dir =  '/Users/tony/Documents/research_projects/mult_colon/crc_fin_dataset/crc/crc_train'
test_data_dir = "/Users/tony/Documents/research_projects/mult_colon/crc_fin_dataset/crc/crc_test"
validation_data_dir = "/Users/tony/Documents/research_projects/mult_colon/crc_fin_dataset/crc/crc_val"

# Image dimensions
IMG_WIDTH, IMG_HEIGHT = 224, 224
input_shape = (IMG_WIDTH, IMG_HEIGHT, 3) # RGB images


# **Data generators**

In [3]:
# Data generators for RGB images

from tensorflow.keras.applications.densenet import preprocess_input

# test_datagen = ImageDataGenerator(rescale=1./255)
# validation_datagen = ImageDataGenerator(rescale=1./255)

validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
testa_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)



# Define data generators for RGB images with augmentation
train_datagen_augmented = ImageDataGenerator(
    #rescale=1./255,
    preprocessing_function = preprocess_input,
    rotation_range=20,  # Rotation angle range
    width_shift_range=0.2,  # Horizontal shift range
    height_shift_range=0.2,  # Vertical shift range
    shear_range=0.2,  # Shear intensity
    zoom_range=0.2,  # Random zoom range
    horizontal_flip=True,  # Randomly flip images horizontally
    vertical_flip=False,  # Do not flip images vertically
    fill_mode='nearest'  # Fill mode for newly created pixels
)

# Generate augmented data for training
train_generator = train_datagen_augmented.flow_from_directory(
    train_data_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=8,
    class_mode='categorical'
  
)
val_datagen = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(IMG_WIDTH,IMG_HEIGHT),
    batch_size = 4,
    class_mode = 'categorical'
)
test_datagen = testa_datagen.flow_from_directory(
    test_data_dir,
    target_size=(IMG_WIDTH,IMG_HEIGHT),
    batch_size = 4,
    class_mode = 'categorical',
    shuffle = False
)



Found 5021 images belonging to 9 classes.
Found 717 images belonging to 9 classes.
Found 1442 images belonging to 9 classes.


In [4]:
class_indices = train_generator.class_indices
print(class_indices)

{'ADI': 0, 'BACK': 1, 'DEB': 2, 'LYM': 3, 'MUC': 4, 'MUS': 5, 'NORM': 6, 'STR': 7, 'TUM': 8}


# **Number of images for each class in the training dataset**

In [5]:
# Count the number of images for each class in the training dataset
#classes = os.listdir(train_data_dir)
classes = [class_name for class_name in os.listdir(train_data_dir) if os.path.isdir(os.path.join(train_data_dir,class_name))]
for class_name in classes:
    class_path = os.path.join(train_data_dir, class_name)
    num_images = len(os.listdir(class_path))
    print(f"Class: {class_name}, Number of images: {num_images}")


Class: ADI, Number of images: 936
Class: MUC, Number of images: 724
Class: BACK, Number of images: 592
Class: LYM, Number of images: 443
Class: NORM, Number of images: 518
Class: DEB, Number of images: 237
Class: MUS, Number of images: 414
Class: TUM, Number of images: 863
Class: STR, Number of images: 294


# **Check the shape of the images in Train Generator**

In [6]:
# Get a batch of images and labels from the train_generator
batch = train_generator.__next__()

# Iterate through the batch to check image shapes
for i in range(len(batch[0])):
    img = batch[0][i]  # Image data
    label = batch[1][i]  # Image label
    
    # Get image shape and channels
    height, width, channels = img.shape
    
    # Display image shape and channels
    print(f"Image {i+1} - Shape: {width}x{height}x{channels}, Label: {label}")


Image 1 - Shape: 224x224x3, Label: [0. 0. 0. 0. 0. 1. 0. 0. 0.]
Image 2 - Shape: 224x224x3, Label: [0. 0. 0. 0. 0. 0. 1. 0. 0.]
Image 3 - Shape: 224x224x3, Label: [1. 0. 0. 0. 0. 0. 0. 0. 0.]
Image 4 - Shape: 224x224x3, Label: [0. 0. 0. 1. 0. 0. 0. 0. 0.]
Image 5 - Shape: 224x224x3, Label: [0. 0. 0. 0. 1. 0. 0. 0. 0.]
Image 6 - Shape: 224x224x3, Label: [1. 0. 0. 0. 0. 0. 0. 0. 0.]
Image 7 - Shape: 224x224x3, Label: [0. 0. 0. 0. 0. 0. 0. 0. 1.]
Image 8 - Shape: 224x224x3, Label: [0. 0. 1. 0. 0. 0. 0. 0. 0.]


# **Number of images for each class in the testing dataset**

In [7]:
# Count the number of images for each class in the testing dataset
classes = os.listdir(test_data_dir)
classes = [class_name for class_name in os.listdir(test_data_dir) if os.path.isdir(os.path.join(train_data_dir,class_name))]
for class_name in classes:
    class_path = os.path.join(test_data_dir, class_name)
    num_images = len(os.listdir(class_path))
    print(f"Class: {class_name}, Number of images: {num_images}")


Class: ADI, Number of images: 268
Class: MUC, Number of images: 208
Class: BACK, Number of images: 170
Class: LYM, Number of images: 128
Class: NORM, Number of images: 149
Class: DEB, Number of images: 68
Class: MUS, Number of images: 119
Class: TUM, Number of images: 247
Class: STR, Number of images: 85


# **Check the shape of the images in Test Generator**

In [8]:
# Get a batch of images and labels from the test_generator
batch = test_datagen.__next__()

# Iterate through the batch to check image shapes
for i in range(len(batch[0])):
    img = batch[0][i]  # Image data
    label = batch[1][i]  # Image label
    
    # Get image shape and channels
    height, width, channels = img.shape
    
    # Display image shape and channels
    print(f"Image {i+1} - Shape: {width}x{height}x{channels}, Label: {label}")


Image 1 - Shape: 224x224x3, Label: [1. 0. 0. 0. 0. 0. 0. 0. 0.]
Image 2 - Shape: 224x224x3, Label: [1. 0. 0. 0. 0. 0. 0. 0. 0.]
Image 3 - Shape: 224x224x3, Label: [1. 0. 0. 0. 0. 0. 0. 0. 0.]
Image 4 - Shape: 224x224x3, Label: [1. 0. 0. 0. 0. 0. 0. 0. 0.]


# **Number of images for each class in the validation dataset**

In [9]:
# Count the number of images for each class in the testing dataset
#classes = os.listdir(validation_data_dir)
classes = [class_name for class_name in os.listdir(train_data_dir) if os.path.isdir(os.path.join(train_data_dir,class_name))]
for class_name in classes:
    class_path = os.path.join(validation_data_dir, class_name)
    num_images = len(os.listdir(class_path))
    print(f"Class: {class_name}, Number of images: {num_images}")


Class: ADI, Number of images: 134
Class: MUC, Number of images: 103
Class: BACK, Number of images: 85
Class: LYM, Number of images: 63
Class: NORM, Number of images: 74
Class: DEB, Number of images: 34
Class: MUS, Number of images: 59
Class: TUM, Number of images: 123
Class: STR, Number of images: 42


# **Check the shape of the images in Validation Generator**

In [10]:
# Get a batch of images and labels from the validation_generator
batch = val_datagen.__next__()

# Iterate through the batch to check image shapes
for i in range(len(batch[0])):
    img = batch[0][i]  # Image data
    label = batch[1][i]  # Image label
    
    # Get image shape and channels
    height, width, channels = img.shape
    
    # Display image shape and channels
    print(f"Image {i+1} - Shape: {width}x{height}x{channels}, Label: {label}")


Image 1 - Shape: 224x224x3, Label: [0. 0. 0. 0. 0. 0. 1. 0. 0.]
Image 2 - Shape: 224x224x3, Label: [0. 0. 0. 0. 0. 0. 1. 0. 0.]
Image 3 - Shape: 224x224x3, Label: [0. 0. 0. 0. 0. 0. 0. 0. 1.]
Image 4 - Shape: 224x224x3, Label: [0. 0. 0. 0. 1. 0. 0. 0. 0.]


# **Check for GPU availability**

In [11]:
# Check for GPU availability
print("GPU is", "available" if tf.config.list_physical_devices('GPU') else "NOT available")

# Set TensorFlow to use the GPU device
if tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(tf.config.list_physical_devices('GPU')[0], True)
    print("GPU device configured")
else:
    print("No GPU device found")

GPU is available
GPU device configured


# **Model checkpoint**

In [33]:
from datetime import datetime
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping,TensorBoard
model_dir = '/Users/tony/Documents/research_projects/mult_colon/model/keras_check/regnety800'

if not os.path.exists(model_dir):
    os.makedirs(model_dir)

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
checkpoint_filename = os.path.join(model_dir,f"model_{timestamp}.keras")

#create an early stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights = True,
    verbose=1
    
)

board_dir= "logs/regnety800/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_log = TensorBoard(
    log_dir = board_dir,
    histogram_freq=1,
    write_graph=True,
    write_images=False
)


# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filename,
                                                 save_weights_only=True,
                                                 save_best_only=True,
                                                 monitor="val_accuracy",   # Monitor validation loss
                                                 mode="max",           # Save the model when validation loss is minimized
                                                 verbose=1)

In [34]:
print(checkpoint_filename)

/Users/tony/Documents/research_projects/mult_colon/model/keras_check/regnety800/model_20260128-114706.keras


# *Architecture*

In [41]:
from tensorflow.keras import models, layers, optimizers

In [12]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from keras_cv_attention_models import regnet
def create_model(summary=True):
    
    # apply transfer learning
    inputs = Input(shape=(224, 224, 3),name='input_layer')
    
    base_model = DenseNet121(weights='imagenet', include_top=False, input_tensor=inputs)
    base_model.trainable = False
    
    # add new classifier layers
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128,activation = 'relu')(x)
    output = Dense(9, activation='softmax')(x)  
    
    # define new model
    model = Model(inputs=inputs,outputs=output)
    
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    if summary:
        print(model.summary())
    return model



# **Model summary**

In [13]:
model = create_model()

2026-01-28 12:28:41.270181: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-01-28 12:28:41.270215: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-01-28 12:28:41.270220: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-01-28 12:28:41.270666: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-28 12:28:41.271028: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_layer (InputLayer)    [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 zero_padding2d (ZeroPaddin  (None, 230, 230, 3)          0         ['input_layer[0][0]']         
 g2D)                                                                                             
                                                                                                  
 conv1/conv (Conv2D)         (None, 112, 112, 64)         9408      ['zero_padding2d[0][0]']      
                                                                                                  
 conv1/bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1/conv[0][0]']      

# **Training starts here**

In [19]:
# Train the model
history = model.fit(
    train_generator, 
    epochs=20,
    validation_data=val_datagen,
    callbacks=[cp_callback,early_stop,tensor_log]
    )

Epoch 1/20


2026-01-27 00:50:20.555358: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


628/628 [==============================] - ETA: 0s - loss: 0.9475 - accuracy: 0.7064     

2026-01-27 00:50:58.005524: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.



Epoch 1: val_accuracy improved from -inf to 0.84937, saving model to /Users/tony/Documents/research_projects/mult_colon/model/keras_check/densenet121/model_20260127-004625.keras
628/628 [==============================] - 51s 71ms/step - loss: 0.9475 - accuracy: 0.7064 - val_loss: 0.4659 - val_accuracy: 0.8494
Epoch 2/20
628/628 [==============================] - ETA: 0s - loss: 0.3436 - accuracy: 0.8992  
Epoch 2: val_accuracy improved from 0.84937 to 0.92050, saving model to /Users/tony/Documents/research_projects/mult_colon/model/keras_check/densenet121/model_20260127-004625.keras
628/628 [==============================] - 40s 64ms/step - loss: 0.3436 - accuracy: 0.8992 - val_loss: 0.2861 - val_accuracy: 0.9205
Epoch 3/20
628/628 [==============================] - ETA: 0s - loss: 0.2429 - accuracy: 0.9233  
Epoch 3: val_accuracy improved from 0.92050 to 0.92469, saving model to /Users/tony/Documents/research_projects/mult_colon/model/keras_check/densenet121/model_20260127-004625.ker

In [14]:
#Loading the model weights
from tensorflow.keras.models import load_model
model = create_model()
model.load_weights('/Users/tony/Documents/research_projects/mult_colon/model/keras_check/densenet121/model_20260127-004625.keras')


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_layer (InputLayer)    [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 zero_padding2d_2 (ZeroPadd  (None, 230, 230, 3)          0         ['input_layer[0][0]']         
 ing2D)                                                                                           
                                                                                                  
 conv1/conv (Conv2D)         (None, 112, 112, 64)         9408      ['zero_padding2d_2[0][0]']    
                                                                                                  
 conv1/bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1/conv[0][0]']    

In [15]:
import time
dummies_input = np.random.rand(1,224,224,3).astype(np.float32)

for _ in range(10):
    model.predict(dummies_input,verbose=0)

#time 
start = time.perf_counter()
model.predict(dummies_input,verbose=0)
end = time.perf_counter()
print(f"Inference time: {(end-start)*1000:.2f} ms")

2026-01-28 12:29:22.642079: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Inference time: 51.04 ms


In [16]:
#Testing of the Image

results = model.evaluate(test_datagen)
print(f"Test Loss: {results[0]:.4f}")
print(f"Test Accuracy: {results[1]*100:.2f}%")

# For multiple metrics:
print("Metric Names:", model.metrics_names)  # Shows order of metrics

2026-01-28 12:29:55.143042: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


361/361 [==============================] - 14s 32ms/step - loss: 0.0765 - accuracy: 0.9730
Test Loss: 0.0765
Test Accuracy: 97.30%
Metric Names: ['loss', 'accuracy']


# Load the dataset for prototype

In [17]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from scipy.spatial.distance import cdist
from sklearn.metrics import accuracy_score
import random

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

base_path = '/Users/tony/Documents/research_projects/mult_colon/CRC_CFHCD_FIN_out'

#  Load Pre-Split Datasets 
def load_dataset_from_split(base_path, subset='train', target_size=(224, 224)):
    """Load images from pre-split train/test folders"""
    subset_path = os.path.join(base_path, subset)
    class_names = sorted(os.listdir(subset_path))
    images, labels = [], []
    
    for class_idx, class_name in enumerate(class_names):
        class_path = os.path.join(subset_path, class_name)
        for img_file in os.listdir(class_path):
            img = image.load_img(os.path.join(class_path, img_file), target_size=target_size)
            img_array = image.img_to_array(img)
            img_array = preprocess_input(img_array)  
            images.append(img_array)
            labels.append(class_idx)

    return np.array(images), np.array(labels), class_names

# Load your pre-split data
train_images, train_labels, class_names = load_dataset_from_split(base_path, 'train')
test_images, test_labels, _ = load_dataset_from_split(base_path, 'test')



In [18]:
#encoder
from tensorflow.keras.models import Model
import tensorflow as tf
import numpy as np
from scipy.spatial.distance import cdist

encoder = Model(
    inputs=model.input,
    outputs=model.layers[-2].output   # Dense(128)
)
encoder.trainable = False


In [19]:
#embedding computing

train_embeddings = encoder.predict(train_images, batch_size=32, verbose=1)
test_embeddings  = encoder.predict(test_images, batch_size=32, verbose=1)

# L2 normalization 
train_embeddings = tf.math.l2_normalize(train_embeddings, axis=1).numpy()
test_embeddings  = tf.math.l2_normalize(test_embeddings, axis=1).numpy()


2026-01-28 12:30:53.518230: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


12/12 [==============================] - 3s 209ms/step


In [20]:
def compute_distances(query_embeddings, prototypes, metric="euclidean"):
    if metric == "euclidean":
        return cdist(query_embeddings, prototypes, metric="euclidean")
    
    elif metric == "cosine":
        # cosine distance = 1 - cosine similarity
        return 1 - np.dot(query_embeddings, prototypes.T)
    
    else:
        raise ValueError("Metric must be 'euclidean' or 'cosine'")


In [21]:
def sample_episode(embeddings, labels, k_shot, n_query):
    support_idx, query_idx = [], []
    
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        selected = np.random.choice(cls_idx, k_shot + n_query, replace=False)
        support_idx.extend(selected[:k_shot])
        query_idx.extend(selected[k_shot:])
    
    return np.array(support_idx), np.array(query_idx)


In [ ]:
import time
from sklearn.metrics import accuracy_score

def evaluate_fewshot(
    train_embeddings,
    train_labels,
    test_embeddings,
    test_labels,
    k_shot=5,
    n_query=15,
    n_episodes=100,
    metric="euclidean"
):
    accuracies = []
    distance_times = []

    for _ in range(n_episodes):
        # Support from TRAIN
        support_idx, _ = sample_episode(
            train_embeddings, train_labels, k_shot, 0
        )

        # Query from TEST
        _, query_idx = sample_episode(
            test_embeddings, test_labels, 0, n_query
        )

        # Prototypes
        prototypes = np.stack([
            np.mean(
                train_embeddings[support_idx][train_labels[support_idx] == cls],
                axis=0
            )
            for cls in np.unique(train_labels)
        ])

        # distance timing ONLY 
        start = time.perf_counter()

        distances = compute_distances(
            test_embeddings[query_idx],
            prototypes,
            metric=metric
        )

        end = time.perf_counter()
        distance_times.append((end - start) / len(query_idx))  # per-query time

        preds = np.argmin(distances, axis=1)
        acc = accuracy_score(test_labels[query_idx], preds)
        accuracies.append(acc)

    return (
        np.mean(accuracies),
        np.std(accuracies),
        np.mean(distance_times)
    )


In [23]:
print("\nFew-shot comparison (same encoder, same episodes):")

for metric in ["euclidean", "cosine"]:
    print(f"\nMetric: {metric.upper()}")
    for k in [1, 5, 10, 20]:
        mean_acc, std_acc, dist_time = evaluate_fewshot(
            train_embeddings,
            train_labels,
            test_embeddings,
            test_labels,
            k_shot=k,
            n_query=15,
            n_episodes=100,
            metric=metric
        )

        print(
            f"{k}-shot: {mean_acc:.2%} ± {std_acc:.2%} | "
            f"Dist time: {dist_time*1e6:.2f} µs/query"
        )



Few-shot comparison (same encoder, same episodes):

Metric: EUCLIDEAN
1-shot: 53.97% ± 8.80% | Dist time: 1.38 µs/query
5-shot: 59.17% ± 8.54% | Dist time: 0.56 µs/query
10-shot: 62.23% ± 11.81% | Dist time: 0.46 µs/query
20-shot: 64.00% ± 9.55% | Dist time: 0.47 µs/query

Metric: COSINE
1-shot: 53.73% ± 9.51% | Dist time: 0.63 µs/query
5-shot: 55.60% ± 8.20% | Dist time: 0.26 µs/query
10-shot: 59.03% ± 9.07% | Dist time: 0.37 µs/query
20-shot: 62.70% ± 9.12% | Dist time: 0.23 µs/query
